# hERG ChEMBL post-processing

This notebook implements the requested post-query cleaning workflow:
- standardize concentration endpoints to nM
- convert IC50/Ki to p-scale (`-log10(M)`) where possible
- preserve and model censoring/inequalities
- separate percent inhibition endpoints
- remove or flag missing/invalid structures
- resolve duplicate compound-assay entries with prioritized selection + median aggregation
- write cleaned outputs and `filtering.log`


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

RAW_PATH = Path("data/herg_raw.csv")
CLEAN_OUTPUT_PATH = Path("data/herg_postprocessed.csv")
PERCENT_OUTPUT_PATH = Path("data/herg_percent_inhibition.csv")
LOG_PATH = Path("filtering.log")

UNIT_TO_NM = {
    "m": 1e9,
    "mm": 1e6,
    "um": 1e3,
    "nm": 1.0,
    "pm": 1e-3,
    "fm": 1e-6,
}
ALLOWED_RELATIONS = {"=", ">", "<", ">=", "<=", "~"}
RELATION_INVERT_FOR_P = {
    "=": "=",
    "~": "~",
    ">": "<",
    ">=": "<=",
    "<": ">",
    "<=": ">=",
}
GROUP_COLS = ["molecule_chembl_id", "assay_chembl_id"]


def clean_text(value):
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, str):
        text = " ".join(value.split())
        return text if text else None
    return str(value)


def normalize_unit(unit_value):
    text = clean_text(unit_value)
    if text is None:
        return None
    return text.lower().replace(chr(181), "u").replace(chr(956), "u").replace(" ", "")


def normalize_relation(rel):
    text = clean_text(rel)
    return text if text in ALLOWED_RELATIONS else None


def to_nm(value_num, unit_norm):
    if pd.isna(value_num) or unit_norm not in UNIT_TO_NM:
        return np.nan
    return float(value_num) * UNIT_TO_NM[unit_norm]


def classify_endpoint(standard_type, standard_units):
    st = (clean_text(standard_type) or "").upper()
    su = (clean_text(standard_units) or "").strip()
    if su == "%" or "INHIB" in st or "PERCENT" in st or st in {"INH", "INHIBITION", "% OF CONTROL"}:
        return "percent_inhibition"
    if st in {"IC50", "KI"}:
        return "ic50_ki"
    if st in {"AC50", "EC50", "IC20", "IC25", "EC10", "LOG IC50", "INFLECTION POINT"}:
        return "other_concentration"
    if normalize_unit(standard_units) in UNIT_TO_NM:
        return "other_concentration"
    return "other_nonstandard"


def censoring_type(relation):
    if relation in {">", ">="}:
        return "right_censored_lower_bound"
    if relation in {"<", "<="}:
        return "left_censored_upper_bound"
    return "uncensored_or_approx"


def first_non_null(series):
    for value in series:
        if pd.notna(value):
            return value
    return np.nan


def smiles_validator():
    try:
        from rdkit import Chem

        def validate(smiles):
            text = clean_text(smiles)
            return bool(text) and Chem.MolFromSmiles(text) is not None

        return validate, "rdkit"
    except Exception:
        pattern = re.compile(r"^[A-Za-z0-9@+\-\[\]\(\)=#$\\\/%\.:\*]+$")

        def validate(smiles):
            text = clean_text(smiles)
            return bool(text) and bool(pattern.fullmatch(text))

        return validate, "regex_fallback_no_rdkit"


df_raw = pd.read_csv(RAW_PATH, low_memory=False)
df = df_raw.copy()

for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].map(clean_text)

df["standard_value_num"] = pd.to_numeric(df["standard_value"], errors="coerce")
df["pchembl_value_num"] = pd.to_numeric(df.get("pchembl_value_num", df["pchembl_value"]), errors="coerce")
df["document_year"] = pd.to_numeric(df["document_year"], errors="coerce").astype("Int64")

df["standard_units_norm"] = df["standard_units"].map(normalize_unit)
df["relation_preserved"] = df["standard_relation"].map(normalize_relation)
df["endpoint_class"] = [classify_endpoint(st, su) for st, su in zip(df["standard_type"], df["standard_units"])]

df["standard_value_nM"] = [to_nm(v, u) for v, u in zip(df["standard_value_num"], df["standard_units_norm"])]
df["standard_value_M"] = df["standard_value_nM"] * 1e-9
positive_molar = df["standard_value_M"] > 0
df["p_activity_bound"] = np.where(positive_molar, -np.log10(df["standard_value_M"]), np.nan)
df["p_activity_relation"] = df["relation_preserved"].map(RELATION_INVERT_FOR_P)
df["censoring_type"] = df["relation_preserved"].map(censoring_type)

std_type_norm = df["standard_type"].fillna("").str.upper()
ic50_ki_mask = std_type_norm.isin({"IC50", "KI"}) & df["p_activity_bound"].notna()
df["p_ic50_ki"] = np.where(ic50_ki_mask, df["p_activity_bound"], np.nan)
df["p_ic50_ki_relation"] = np.where(ic50_ki_mask, df["p_activity_relation"], None)

df["is_percent_inhibition"] = df["endpoint_class"].eq("percent_inhibition")
df["pseudo_ic50_method"] = np.where(df["is_percent_inhibition"], "not_computed_no_curve_data", None)

validate_smiles, smiles_method = smiles_validator()
df["smiles_missing"] = df["canonical_smiles"].isna()
df["smiles_is_valid"] = df["canonical_smiles"].map(validate_smiles)
df["smiles_validation_method"] = smiles_method

has_measure = df["standard_type"].notna() | df["standard_value"].notna() | df["standard_units"].notna()
core_df = df[df["assay_chembl_id"].notna() & df["molecule_chembl_id"].notna() & has_measure].copy()
filtered_df = core_df[(~core_df["smiles_missing"]) & core_df["smiles_is_valid"]].copy()

filtered_std = filtered_df["standard_type"].fillna("").str.upper()
filtered_df["priority_score"] = (
    8 * filtered_std.isin({"IC50", "KI"}).astype(int)
    + 4 * filtered_df["assay_organism"].fillna("").str.contains(r"homo sapiens|human", case=False, regex=True).astype(int)
    + 2 * filtered_df["relation_preserved"].eq("=").astype(int)
    + filtered_df["standard_value_nM"].notna().astype(int)
)

max_priority = filtered_df.groupby(GROUP_COLS)["priority_score"].transform("max")
selected_df = filtered_df[filtered_df["priority_score"] == max_priority].copy()

numeric_cols = [
    "standard_value_num",
    "standard_value_nM",
    "standard_value_M",
    "p_activity_bound",
    "p_ic50_ki",
    "pchembl_value_num",
]

selected_sorted = selected_df.sort_values(
    by=GROUP_COLS + ["priority_score"],
    ascending=[True, True, False],
    kind="stable",
)
representative = selected_sorted.groupby(GROUP_COLS, as_index=False).agg(first_non_null)
medians = selected_df.groupby(GROUP_COLS, as_index=False)[numeric_cols].median(numeric_only=True)
medians = medians.rename(columns={c: f"{c}_median" for c in numeric_cols})
count_selected = selected_df.groupby(GROUP_COLS, as_index=False).size().rename(columns={"size": "n_selected_measurements"})
count_total = filtered_df.groupby(GROUP_COLS, as_index=False).size().rename(columns={"size": "n_total_measurements_in_group"})

cleaned_df = representative.merge(medians, on=GROUP_COLS, how="left")
cleaned_df = cleaned_df.merge(count_selected, on=GROUP_COLS, how="left")
cleaned_df = cleaned_df.merge(count_total, on=GROUP_COLS, how="left")

for col in numeric_cols:
    cleaned_df[col] = cleaned_df[f"{col}_median"].combine_first(cleaned_df[col])

cleaned_df = cleaned_df.sort_values(GROUP_COLS, kind="stable").reset_index(drop=True)
percent_df = cleaned_df[cleaned_df["is_percent_inhibition"]].copy()

cleaned_df.to_csv(CLEAN_OUTPUT_PATH, index=False)
percent_df.to_csv(PERCENT_OUTPUT_PATH, index=False)

counts = {
    "raw_rows": int(len(df_raw)),
    "core_rows": int(len(core_df)),
    "removed_missing_smiles": int(core_df["smiles_missing"].sum()),
    "removed_invalid_smiles": int((~core_df["smiles_missing"] & ~core_df["smiles_is_valid"]).sum()),
    "rows_after_structure_filter": int(len(filtered_df)),
    "rows_converted_nM": int(filtered_df["standard_value_nM"].notna().sum()),
    "rows_converted_pIC50_pKi": int(filtered_df["p_ic50_ki"].notna().sum()),
    "percent_inhibition_rows": int(filtered_df["is_percent_inhibition"].sum()),
    "censored_rows": int(filtered_df["relation_preserved"].isin({">", "<", ">=", "<="}).sum()),
    "rows_after_duplicate_resolution": int(len(cleaned_df)),
    "duplicate_rows_collapsed": int(len(filtered_df) - len(cleaned_df)),
    "smiles_validation_method": smiles_method,
}

log_lines = [
    "Data cleaning and standardization log",
    "=====================================",
    "",
    "Rules implemented:",
    "1. Standardized concentration units to nM where units were in {M, mM, uM/um, nM, pM, fM}.",
    "2. Converted IC50/Ki rows to pIC50/pKi using -log10(M) when concentration values were available.",
    "3. Preserved inequality relations and labeled censoring; p-scale relations are inverted from concentration scale.",
    "4. Treated percent inhibition endpoints as a separate assay class and did not compute pseudo-IC50 without curve data.",
    "5. Removed rows with missing/invalid SMILES from the cleaned output, while retaining flags for auditability.",
    "6. Resolved duplicates per compound-assay by prioritized selection (IC50/Ki, human assay, uncensored, numeric concentration) and median aggregation for numeric values.",
    "",
    "Counts:",
    f"- N raw rows: {counts['raw_rows']}",
    f"- Core rows (assay + molecule + measure present): {counts['core_rows']}",
    f"- Removed for missing SMILES: {counts['removed_missing_smiles']}",
    f"- Removed for invalid SMILES: {counts['removed_invalid_smiles']}",
    f"- Rows after structure filter: {counts['rows_after_structure_filter']}",
    f"- Converted to nM: {counts['rows_converted_nM']}",
    f"- Converted to pIC50/pKi: {counts['rows_converted_pIC50_pKi']}",
    f"- Percent inhibition rows flagged: {counts['percent_inhibition_rows']}",
    f"- Censored rows preserved: {counts['censored_rows']}",
    f"- Rows after duplicate resolution: {counts['rows_after_duplicate_resolution']}",
    f"- Duplicate rows collapsed: {counts['duplicate_rows_collapsed']}",
    f"- SMILES validation method: {counts['smiles_validation_method']}",
    "",
    "Outputs:",
    "- data/herg_postprocessed.csv",
    "- data/herg_percent_inhibition.csv",
    "- filtering.log",
]
LOG_PATH.write_text("\n".join(log_lines) + "\n", encoding="utf-8")

print("Created:", CLEAN_OUTPUT_PATH)
print("Created:", PERCENT_OUTPUT_PATH)
print("Created:", LOG_PATH)
for key, value in counts.items():
    print(f"{key}: {value}")


Created: data\herg_postprocessed.csv
Created: data\herg_percent_inhibition.csv
Created: filtering.log
raw_rows: 2861
core_rows: 2861
removed_missing_smiles: 1
removed_invalid_smiles: 0
rows_after_structure_filter: 2860
rows_converted_nM: 2149
rows_converted_pIC50_pKi: 1687
percent_inhibition_rows: 439
censored_rows: 650
rows_after_duplicate_resolution: 2661
duplicate_rows_collapsed: 199
smiles_validation_method: regex_fallback_no_rdkit
